# Wine quality: hyperparameter sweep → Model Registry → champion alias

The original lab trained one ElasticNet model with fixed settings (`alpha=0.5`, `l1_ratio=0.5`).
This version turns it into a small model-management pipeline:

1. **Sweep:** train one model for every `alpha` × `l1_ratio` combination. Each one is a *child run* nested under one *parent run*, so the whole sweep is grouped together in the MLflow UI.
2. **Log more than numbers:** each child run logs its parameters, metrics, a residuals plot, and the model with a signature and an input example. The parent run logs the dataset.
3. **Pick the best run in code:** use `mlflow.search_runs` to rank the child runs by RMSE instead of picking one by eye.
4. **Register and promote:** register the best model in the Model Registry. It gets the `champion` alias only if it beats the current champion; otherwise it is labelled `challenger`.
5. **Serve by alias:** `mlflow models serve -m "models:/ElasticnetWineModel@champion"` always serves the current champion, with no run IDs to copy (see the last cell).

In [1]:
import itertools
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from mlflow.models import infer_signature

from wine_data import CSV_URL, TARGET, load_wine_data, split_wine_data

warnings.filterwarnings("ignore")

# Store runs in a local SQLite database so the Model Registry works
mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_NAME = "wine-elasticnet-sweep"
MODEL_NAME = "ElasticnetWineModel"
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()

# Hyperparameter grid: every combination becomes one child run
ALPHAS = [0.01, 0.1, 0.5, 1.0]
L1_RATIOS = [0.1, 0.5, 0.9]

2026/10/29 02:52:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/29 02:52:54 INFO mlflow.store.db.utils: Updating database tables


2026/10/29 02:52:56 INFO mlflow.tracking.fluent: Experiment with name 'wine-elasticnet-sweep' does not exist. Creating a new experiment.


### Load the data
The split uses a fixed `random_state`, so `test_endpoint.py` can rebuild exactly the same test set later.

In [2]:
data = load_wine_data()
train_x, test_x, train_y, test_y = split_wine_data(data)
print(f"train: {train_x.shape}, test: {test_x.shape}")

train: (1199, 11), test: (400, 11)


In [3]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2


def residual_plot(actual, pred, title):
    """Predicted vs. residual scatter, logged as an image artifact on each run."""
    residuals = np.ravel(actual) - np.ravel(pred)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(pred, residuals, alpha=0.4, s=12)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("Predicted quality")
    ax.set_ylabel("Residual (actual - predicted)")
    ax.set_title(title)
    fig.tight_layout()
    plt.close(fig)
    return fig

### Step 1–2: Run the sweep as nested runs

In [4]:
with mlflow.start_run(run_name="elasticnet-sweep") as parent:
    mlflow.log_input(
        mlflow.data.from_pandas(data, source=CSV_URL, name="winequality-red", targets=TARGET),
        context="training",
    )
    mlflow.log_params({"alphas": ALPHAS, "l1_ratios": L1_RATIOS, "n_candidates": len(ALPHAS) * len(L1_RATIOS)})

    for alpha, l1_ratio in itertools.product(ALPHAS, L1_RATIOS):
        run_name = f"alpha={alpha}, l1_ratio={l1_ratio}"
        with mlflow.start_run(run_name=run_name, nested=True):
            lr = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
            lr.fit(train_x, train_y)

            predicted_qualities = lr.predict(test_x)
            rmse, mae, r2 = eval_metrics(test_y, predicted_qualities)

            mlflow.log_params({"alpha": alpha, "l1_ratio": l1_ratio})
            mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
            mlflow.log_figure(residual_plot(test_y, predicted_qualities, run_name), "residuals.png")
            mlflow.sklearn.log_model(
                lr,
                name="model",
                signature=infer_signature(train_x, lr.predict(train_x)),
                input_example=test_x.head(3),
            )
            print(f"{run_name:<28} RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")

print(f"\nParent run ID: {parent.info.run_id}")

alpha=0.01, l1_ratio=0.1     RMSE=0.6742  MAE=0.5166  R2=0.3560


alpha=0.01, l1_ratio=0.5     RMSE=0.6779  MAE=0.5191  R2=0.3490


alpha=0.01, l1_ratio=0.9     RMSE=0.6803  MAE=0.5214  R2=0.3442


alpha=0.1, l1_ratio=0.1      RMSE=0.7129  MAE=0.5462  R2=0.2799


alpha=0.1, l1_ratio=0.5      RMSE=0.7309  MAE=0.5615  R2=0.2431


...

alpha=0.5, l1_ratio=0.5      RMSE=0.7932  MAE=0.6272  R2=0.1086


alpha=0.5, l1_ratio=0.9      RMSE=0.8324  MAE=0.6669  R2=0.0183


alpha=1.0, l1_ratio=0.1      RMSE=0.7702  MAE=0.6019  R2=0.1596


alpha=1.0, l1_ratio=0.5      RMSE=0.8328  MAE=0.6682  R2=0.0173


alpha=1.0, l1_ratio=0.9      RMSE=0.8329  MAE=0.6718  R2=0.0171

Parent run ID: 5c0be927dc4d416f89f78a193b3d29ab


### Step 3: Pick the best run in code
Rank only the child runs of *this* sweep (`tags.mlflow.parentRunId`) by RMSE, lowest first.

In [5]:
ranked = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'",
    order_by=["metrics.rmse ASC"],
)
best = ranked.iloc[0]
best_run_id, best_rmse = best.run_id, best["metrics.rmse"]

# Record the winner on the parent run so it is visible in the UI
client.log_metric(parent.info.run_id, "best_rmse", best_rmse)
client.set_tag(parent.info.run_id, "best_run_id", best_run_id)

ranked[["tags.mlflow.runName", "params.alpha", "params.l1_ratio", "metrics.rmse", "metrics.mae", "metrics.r2"]].head()

,tags.mlflow.runName,params.alpha,params.l1_ratio,metrics.rmse,metrics.mae,metrics.r2
0,"alpha=0.01, l1_ratio=0.1",0.01,0.1,0.674172,0.516593,0.356016
1,"alpha=0.01, l1_ratio=0.5",0.01,0.5,0.677856,0.519056,0.348959
2,"alpha=0.01, l1_ratio=0.9",0.01,0.9,0.680342,0.521439,0.344175
3,"alpha=0.1, l1_ratio=0.1",0.1,0.1,0.712883,0.546220,0.279938
4,"alpha=0.1, l1_ratio=0.5",0.1,0.5,0.730900,0.561549,0.243081


### Step 4: Register the best model and decide champion vs. challenger
Every sweep registers its best model as a new version of `ElasticnetWineModel`. It takes the `champion` alias only if its RMSE is lower than the current champion's; otherwise it gets the `challenger` alias and the champion stays where it is. Running the notebook again with the same grid therefore produces a challenger, not a new champion.

In [6]:
new_version = mlflow.register_model(f"runs:/{best_run_id}/model", MODEL_NAME)
client.set_model_version_tag(MODEL_NAME, new_version.version, "rmse", f"{best_rmse:.4f}")
client.set_model_version_tag(MODEL_NAME, new_version.version, "sweep_run_id", parent.info.run_id)

try:
    current = client.get_model_version_by_alias(MODEL_NAME, "champion")
    current_rmse = client.get_run(current.run_id).data.metrics["rmse"]
except MlflowException:
    current, current_rmse = None, float("inf")

if best_rmse < current_rmse:
    client.set_registered_model_alias(MODEL_NAME, "champion", new_version.version)
    previous = f"version {current.version} (RMSE {current_rmse:.4f})" if current else "none"
    print(f"Promoted version {new_version.version} (RMSE {best_rmse:.4f}) to champion. Previous champion: {previous}")
else:
    client.set_registered_model_alias(MODEL_NAME, "challenger", new_version.version)
    print(
        f"Version {new_version.version} (RMSE {best_rmse:.4f}) did not beat champion version "
        f"{current.version} (RMSE {current_rmse:.4f}); labelled it challenger"
    )

Successfully registered model 'ElasticnetWineModel'.
2026/10/29 02:54:37 WARNING mlflow.tracking._model_registry.fluent: Run with id ee176a18a9cd4fdca61e20ce0b2c0546 has no artifacts at artifact path 'model', registering model based on models:/m-2a9ebb744c304dbbb978d8b5f4be7625 instead


Promoted version 1 (RMSE 0.6742) to champion. Previous champion: none


Created version '1' of model 'ElasticnetWineModel'.


### Check: load the champion by alias
This is the same URI the model server uses in the next step.

In [7]:
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print(f"Champion is version {champion.version} from run {champion.run_id}")
print("Predictions for the first 5 test rows:", np.round(champion_model.predict(test_x.head()), 3).tolist())

Champion is version 1 from run ee176a18a9cd4fdca61e20ce0b2c0546
Predictions for the first 5 test rows: [5.925, 5.088, 5.764, 6.409, 5.443]


### Step 5: Serve the champion and test the endpoint
The server blocks its terminal, so run these in a **separate terminal** from the `mlflow_lab1` folder, with the virtual environment activated:

```
mlflow models serve -m "models:/ElasticnetWineModel@champion" --env-manager=local -h 127.0.0.1 -p 5001
```

Then, in another terminal (also activated):

```
python test_endpoint.py
```

It prints `PASS` when the served predictions match the champion loaded from the registry. To browse the runs, plots and registered versions, run `mlflow ui` and open http://127.0.0.1:5000.